code adapted from :
* https://github.com/spmallick/learnopencv/blob/master/Handwritten_Text_Recognition_using_OCR/Fine_Tune_TrOCR_Handwritten.ipynb
* https://github.com/spmallick/learnopencv/blob/master/Handwritten_Text_Recognition_using_OCR/Pretrained_Model_Inference.ipynb
* https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Evaluating_TrOCR_base_handwritten_on_the_IAM_test_set.ipynb
* https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Fine_tune_TrOCR_on_IAM_Handwriting_Database_using_native_PyTorch.ipynb
* https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Fine_tune_TrOCR_on_IAM_Handwriting_Database_using_Seq2SeqTrainer.ipynb

In [1]:
from huggingface_hub import login
import os
access_token = os.getenv("RUNPOD_SECRET_hf_token")
login=login(token=access_token)

In [8]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer, TrOCRForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

processor = TrOCRProcessor.from_pretrained("ds1219/Pandithaya-Printed-Words-128", use_fast=True, size=128)
tokenizer = AutoTokenizer.from_pretrained("ds1219/Pandithaya-Printed-Words-128")
processor.tokenizer = tokenizer

model = VisionEncoderDecoderModel.from_pretrained("ds1219/Pandithaya-Printed-Words-128")

In [9]:

dora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.01,
    use_dora=True,
    target_modules=["query", "value"],
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.01,
    use_dora=True,
    target_modules=["q_proj", "v_proj"]
)

# Apply LoRA to encoder and decoder
model.encoder = get_peft_model(model.encoder, dora_config)
model.decoder = get_peft_model(model.decoder, lora_config)

In [10]:
# Set special tokens and config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 32
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 0.8
model.config.num_beams = 2
model.config.image_size = 128

In [11]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable_params} / Total params: {total_params}")

Trainable params: 1124352 / Total params: 316342784


In [12]:
from peft import PeftModel
import os

def save_peft_model(model, output_dir, epoch=None):
    if epoch is not None:
        save_dir = os.path.join(output_dir, f"epoch_{epoch}")
    else:
        save_dir = output_dir
    
    os.makedirs(save_dir, exist_ok=True)
    
    encoder_dir = os.path.join(save_dir, "encoder_adapters")
    os.makedirs(encoder_dir, exist_ok=True)
    model.encoder.save_pretrained(encoder_dir)
    
    decoder_dir = os.path.join(save_dir, "decoder_adapters")
    os.makedirs(decoder_dir, exist_ok=True)
    model.decoder.save_pretrained(decoder_dir)
    
    model.config.save_pretrained(save_dir)
    
    print(f"PEFT model adapters saved to {save_dir}")
    return save_dir

def load_peft_model(base_model_path, adapter_path):
    model = VisionEncoderDecoderModel.from_pretrained(base_model_path)
    
    dora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.01,
        use_dora=True,
        target_modules=["query", "value"],
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.01,
        bias="none",
        use_dora=True,
        target_modules=["q_proj", "v_proj"]
    )
    
    model.encoder = get_peft_model(model.encoder, dora_config)
    model.decoder = get_peft_model(model.decoder, lora_config)
    
    encoder_adapter_path = os.path.join(adapter_path, "encoder_adapters")
    decoder_adapter_path = os.path.join(adapter_path, "decoder_adapters")
    
    if os.path.exists(encoder_adapter_path):
        print(f"Loading encoder adapters from {encoder_adapter_path}")
        model.encoder.load_adapter(encoder_adapter_path)
    
    if os.path.exists(decoder_adapter_path):
        print(f"Loading decoder adapters from {decoder_adapter_path}")
        model.decoder.load_adapter(decoder_adapter_path)
    
    return model

In [13]:
import pandas as pd

def import_dataset(path):
  df = pd.read_csv(path, sep='\t', usecols=[0,1])
  #df = pd.read_csv(path, sep='\t', usecols=[0,1])
  df.rename(columns={'file': 'file_name', 'labels': 'text'}, inplace=True)
  #df = df.iloc[1:]
  return(df)

In [14]:
train_df = import_dataset('/workspace/pandith/synth-handwritten/train.tsv')

#train_df = train_df.truncate(after=000)
train_df.head()

,file_name,text
0,9194.jpg,මාරුවුණු
1,9195.jpg,නතරවු
2,9196.jpg,විරාමයට
3,9197.jpg,නායයෑම
4,9198.jpg,ප්‍රසන්නභාවය


In [15]:
test_df = import_dataset('/workspace/pandith/synth-handwritten/test.tsv')

#train_df = train_df.truncate(after=5000)
test_df.head()

,file_name,text
0,0.jpg,සියල්ලනටම
1,3.jpg,ගැලපෙන්නෙ
2,4.jpg,ෂෝපහාෂය
3,5.jpg,ජේත්තුවට
4,7.jpg,සව්සිරිපායේදී


In [16]:
import os
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import numpy as np
import random
import skimage.filters
from skimage.util import random_noise
from tqdm import tqdm
import colorsys
import cv2
from skimage.filters import threshold_sauvola
from skimage import img_as_ubyte

def to_apply():
    return random.choice([True, False])


def noisy(noise_typ, image):
    img_array = np.array(image)

    if noise_typ == "gauss":
        noisy_array = random_noise(img_array, mode="gaussian")
    elif noise_typ == "s&p":
        noisy_array = random_noise(img_array, mode="s&p")
    elif noise_typ == "poisson":
        noisy_array = random_noise(img_array, mode="poisson")

    noisy_array = (noisy_array * 255).astype(np.uint8)

    return Image.fromarray(noisy_array)


# Function to apply random enhancements to an image
def apply_random_enhancements(image):

    rotation = random.randint(-10, 10)
    image = image.rotate(rotation, expand=True, fillcolor=(255, 255, 255))

    if to_apply():
        # Randomly adjust color balance
        color_factor = random.uniform(1, 5)
        enhancer = ImageEnhance.Color(image)
        image = enhancer.enhance(color_factor)

    if to_apply():
        # Apply a randomly colored filter
        h = random.uniform(0, 1)
        s = random.uniform(0.43, 1) 
        l = random.uniform(0.2, 0.6)

        r, g, b = colorsys.hls_to_rgb(h, l, s)

        r, g, b = int(r * 255), int(g * 255), int(b * 255)

        color_layer = Image.new("RGB", image.size, (r, g, b))

        alpha = random.uniform(0.1, 0.6)
        image = Image.blend(image, color_layer, alpha)

    if to_apply():
        brightness_factor = random.uniform(0.5, 1.5)
        enhancer = ImageEnhance.Brightness(image)
        image = enhancer.enhance(brightness_factor)

    if to_apply():
        sharpness_factor = random.uniform(0.5, 1.5)
        enhancer = ImageEnhance.Sharpness(image)
        image = enhancer.enhance(sharpness_factor)

    if to_apply():
        contrast_factor = random.uniform(0.5, 1.5)
        enhancer = ImageEnhance.Contrast(image)
        image = enhancer.enhance(contrast_factor)

    if to_apply():
        blur_factor = random.uniform(0, 2)
        image = image.filter(ImageFilter.GaussianBlur(blur_factor))

    if to_apply():
        noise = random.choice(["gauss", "poisson", "s&p"])
        image = noisy(noise, image)

    return image


def binarize_image(image):

    image = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    window_size = 25
    sauvola_thresh = threshold_sauvola(gray, window_size=window_size)
    binary = (gray > sauvola_thresh).astype(np.uint8) * 255
    binary_rgb = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB,)
    return binary_rgb

def preprocess(img):
    img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    img = binarize_image(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    return img

In [18]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, eval=False, max_target_length=128,):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length
        self.eval = eval

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text
        file_name = self.df['file_name'][idx]
        if ".jpg" not in file_name:
          file_name = (file_name.split(".")[0]) + ".jpg"
        text = self.df['text'][idx]

        if text != "":
            # prepare image (i.e. resize + normalize)
            image = Image.open(self.root_dir + file_name).convert("RGB")
            image = preprocess(image)
            #if self.eval:
            #    image = apply_random_enhancements(image)
            pixel_values = self.processor(image, return_tensors="pt").pixel_values
            # add labels (input_ids) by encoding the text
            labels = self.processor.tokenizer(text,
                                              padding="max_length",
                                              max_length=self.max_target_length).input_ids
            # important: make sure that PAD tokens are ignored by the loss function
            labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
    
            encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
            return encoding

Let's initialize the training and evaluation datasets:

In [19]:

train_dataset = IAMDataset(root_dir='/workspace/pandith/synth-handwritten/train/',
                           df=train_df,
                           processor=processor, eval=True)

eval_dataset = IAMDataset(root_dir='/workspace/pandith/synth-handwritten/test/',
                           df=test_df,
                           processor=processor)

In [20]:
print("Number of training examples:", len(train_dataset))
print("Number of validation examples:", len(eval_dataset))

Number of training examples: 29696
Number of validation examples: 7422


Let's verify an example from the training dataset:

In [21]:
encoding = train_dataset[1]
for k,v in encoding.items():
  print(k, v.shape)

pixel_values torch.Size([3, 128, 128])
labels torch.Size([128])


We can also check the original image and decode the labels:

In [22]:
labels = encoding['labels']
labels[labels == -100] = processor.tokenizer.pad_token_id
label_str = processor.decode(labels, skip_special_tokens=True)
print(label_str)

නතරවු


Let's create corresponding dataloaders:

In [23]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
eval_dataloader = DataLoader(eval_dataset, batch_size=32)

## Train a model

Here, we initialize the TrOCR model from its pretrained weights. Note that the weights of the language modeling head are already initialized from pre-training, as the model was already trained to generate text during its pre-training stage. Refer to the paper for details.

In [24]:
from transformers import VisionEncoderDecoderModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.compile()

Importantly, we need to set a couple of attributes, namely:
* the attributes required for creating the `decoder_input_ids` from the `labels` (the model will automatically create the `decoder_input_ids` by shifting the `labels` one position to the right and prepending the `decoder_start_token_id`, as well as replacing ids which are -100 by the pad_token_id)
* the vocabulary size of the model (for the language modeling head on top of the decoder)
* beam-search related parameters which are used when generating text.

We will evaluate the model on the Character Error Rate (CER), which is available in HuggingFace Datasets (see [here](https://huggingface.co/metrics/cer)).

In [25]:
import evaluate

cer_metric = evaluate.load("cer")

In [26]:
def compute_cer(pred_ids, label_ids):
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return cer

In [27]:
import torch.optim.lr_scheduler as lr_scheduler

def get_current_lr(optimizer):
    """Get the current learning rate from the optimizer."""
    for param_group in optimizer.param_groups:
        return param_group['lr']

In [28]:
from copy import deepcopy

In [30]:
from torch.optim import AdamW
from tqdm.notebook import tqdm
from torch.utils.tensorboard import SummaryWriter

optimizer = AdamW(model.parameters(), lr=5e-5)

writer = SummaryWriter(log_dir="/workspace/pandith/tensorboard_logs")


scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',
    factor=0.5,
    patience=2,
    threshold=0.01,
    threshold_mode='rel',
    min_lr=1e-6
)

best_cer = float('inf')
global_step = 0
for epoch in range(10):
   epoch += 10
   model.to(device)
   model.train()
   train_loss = 0.0
   for batch_idx, batch in enumerate(tqdm(train_dataloader)):
      for k,v in batch.items():
        batch[k] = v.to(device)
      outputs = model(**batch,interpolate_pos_encoding=True)
      loss = outputs.loss
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      train_loss += loss.item()
      global_step += 1

      # Log training loss every 50 steps
      if global_step % 50 == 0:
          avg_loss = train_loss / (batch_idx + 1)
          writer.add_scalar('Loss/train', avg_loss, global_step)

   avg_train_loss = train_loss/len(train_dataloader)
   print(f"Loss after epoch {epoch}:", avg_train_loss)
   print(f"Current learning rate: {get_current_lr(optimizer)}")
   writer.add_scalar('Loss/epoch_train', avg_train_loss, epoch)
   
   save_peft_model(model, "/workspace/pandith/model/checkpoint", epoch)
   
   model2 = deepcopy(model)
   model2.encoder = model2.encoder.merge_and_unload()
   model2.decoder = model2.decoder.merge_and_unload()
   model2.to(device)

   model.eval()
   valid_cer = 0.0
   with torch.no_grad():
     for batch in tqdm(eval_dataloader):
       for k,v in batch.items():
         batch[k] = v.to(device)
       outputs = model2.generate(batch["pixel_values"],interpolate_pos_encoding=True)
       cer = compute_cer(pred_ids=outputs, label_ids=batch["labels"])
       valid_cer += cer 
   
   avg_valid_cer = valid_cer / len(eval_dataloader)
   print("Validation CER:", avg_valid_cer)
   writer.add_scalar('CER/valid', avg_valid_cer, epoch)

   del(model2)
   torch.cuda.empty_cache()
   
   old_lr = get_current_lr(optimizer)
   
   scheduler.step(avg_valid_cer)
   
   new_lr = get_current_lr(optimizer)
   if new_lr != old_lr:
       print(f"Learning rate reduced from {old_lr} to {new_lr}")
       writer.add_scalar('LR/reduced', new_lr, epoch)
   
   if avg_valid_cer < best_cer:
     best_cer = avg_valid_cer
     print(f"New best CER: {best_cer}! Saving model...")
     writer.add_scalar('CER/best', best_cer, epoch)
     
      #Save PEFT adapters for continued training
     save_peft_model(model, "/workspace/pandith/model/best")
     
      #Also save merged model for inference
     model_merged = deepcopy(model)
     model_merged.encoder = model_merged.encoder.merge_and_unload()
     model_merged.decoder = model_merged.decoder.merge_and_unload()
     model_merged.save_pretrained("/workspace/pandith/model/best_merged")

writer.close()

  0%|          | 0/3712 [00:00<?, ?it/s]

  0%|          | 0/3712 [00:00<?, ?it/s]

Loss after epoch 10: 2.4975009586597823
Current learning rate: 5e-05
PEFT model adapters saved to /workspace/pandith/model/checkpoint/epoch_10


  0%|          | 0/3712 [00:00<?, ?it/s]

Loss after epoch 10: 2.4975009586597823
Current learning rate: 5e-05
PEFT model adapters saved to /workspace/pandith/model/checkpoint/epoch_10


...

Loss after epoch 12: 2.4037044673419463
Current learning rate: 5e-05
PEFT model adapters saved to /workspace/pandith/model/checkpoint/epoch_12


  0%|          | 0/928 [00:00<?, ?it/s]

Validation CER: 0.4565506871842202


  0%|          | 0/3712 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [63]:
import torch
torch.cuda.empty_cache()

In [314]:
model.to(device)
model.eval()

print("Ground Truth vs Predicted Text:")
with torch.no_grad():
    for batch in tqdm(train_dataloader):
        # Move inputs to device
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]
        
        # Generate predictions
        outputs = model2.generate(pixel_values, interpolate_pos_encoding=True)
        
        # Decode predictions
        pred_texts = processor.batch_decode(outputs, skip_special_tokens=True)
        
        # Fix ground truth labels - replace -100 with pad_token_id before decoding
        labels_for_decoding = labels.clone()
        labels_for_decoding[labels_for_decoding == -100] = processor.tokenizer.pad_token_id
        ground_truth_texts = processor.batch_decode(labels_for_decoding, skip_special_tokens=True)
        
        # Print ground truth and predictions
        for gt, pred in zip(ground_truth_texts, pred_texts):
            print(f"Ground Truth: {gt}")
            print(f"Predicted: {pred}")
            print("-" * 50)

Ground Truth vs Predicted Text:


  0%|          | 0/67 [00:00<?, ?it/s]

Ground Truth: නික්මේ
Predicted: නික්ම
--------------------------------------------------
Ground Truth: අශීර්වාදළත්
Predicted: චාගෙන්ගේගේ
--------------------------------------------------
Ground Truth: මෙහෙශිය
Predicted: මෙෞශි
--------------------------------------------------
Ground Truth: ප්රමාදකාරී
Predicted: සුවතාා
--------------------------------------------------
Ground Truth: ඉණ්ද්රජාලකථණය
Predicted: ඉජණ්ජණණය
--------------------------------------------------
Ground Truth: ජාලයක
Predicted: ජාලයක
--------------------------------------------------
Ground Truth: රිදවා
Predicted: රිද
--------------------------------------------------
Ground Truth: සිසුවියන්ද
Predicted: සිසුවියදද
--------------------------------------------------
Ground Truth: ශිවුරළු
Predicted: ශීුවේ
--------------------------------------------------
Ground Truth: සුවිසුද්ධ
Predicted: සුවටට
--------------------------------------------------
Ground Truth: සූරතිස්ස
Predicted: අරයය
-------------------------------------

KeyboardInterrupt: 

In [18]:
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import HfApi, HfFolder, Repository, create_repo, upload_folder


upload_folder(
    repo_id='ds1219/pandith-synth-real',
    folder_path='/workspace/pandith/synth+real model/best_merged',
    path_in_repo=".",  # Upload everything from model_save_path to the root
    commit_message="Upload model and tokenizer"
)

CommitInfo(commit_url='https://huggingface.co/ds1219/pandith-synth-real/commit/83a67201cbc3fcfc3ac9d9831b15678839b9569b', commit_message='Upload model and tokenizer', commit_description='', oid='83a67201cbc3fcfc3ac9d9831b15678839b9569b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ds1219/pandith-synth-real', endpoint='https://huggingface.co', repo_type='model', repo_id='ds1219/pandith-synth-real'), pr_revision=None, pr_num=None)